# Methods Extension: Full‑Grid Tuning + LOO Evaluation

This notebook runs subject‑level LOO evaluation with **inner 5‑fold tuning** for:
- Logistic Regression (L1/L2)
- SVM (linear / RBF)
- Random Forest

We report **Accuracy** and **Cohen’s d (V2−V1)** in one table, for both:
- Full feature set
- Entropy/IG‑selected feature set

We also fit a simple **linear calibration** (score → clinical value) on training folds to predict **FARS** and **SARA** (per‑visit and ΔV2−V1), and report regression metrics.

Only one CSV is saved: `results/feature_importance_ml_extensions.csv`.


In [1]:
# Purpose: imports, config, and paths
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')

BUNDLE_ROOT = Path('..').resolve()
os.chdir(BUNDLE_ROOT)
print('Working directory:', Path.cwd())

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = Path('data/processed/melbourne_frda_merged.csv')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INNER_FOLDS = 5
K_LIST = [500, 1000, 2000, 5000]
N_PERM = 10  # permutation importance per outer fold


Working directory: /Users/robertwang/Documents/New project/baseline_composite_biomakers


In [2]:
# Purpose: load wide data and detect subject column
melb_wide = pd.read_csv(DATA_PATH)

for _c in ['melb_id', 'ID', 'subject', 'subject_id']:
    if _c in melb_wide.columns:
        subject_col = _c
        break
else:
    raise KeyError('No subject id column found')

print('Subjects:', melb_wide[subject_col].nunique())
print('Columns:', len(melb_wide.columns))


Subjects: 26
Columns: 69


In [ ]:
# feature definitions and combinations
background = [
    'age', 'age_at_onset', 'dur', 'sex',
    'GAA1', 'GAA2'
]

structural = [
    'SCP', 'MCP', 'ICP',
    'Midbrain', 'Pons', 'Medulla',
    'AntCBLM', 'SupPostCBLM', 'InfPostCBLM', 'FlocCBLM', 'VermisCBLM'
]

structural_ext = structural + ['CSA_C1', 'CSA_C2', 'ECC_C1', 'ECC_C2']

diffusion = [
    'FASCP', 'FAMCP', 'FAICP',
    'MDSCP', 'MDMCP', 'MDICP',
    'ADSCP', 'ADMCP', 'ADICP',
    'RDSCP', 'RDMCP', 'RDICP'
]

combinations = [
    {'name': 'background_only', 'features': background},
    {'name': 'structural_only', 'features': structural},
    {'name': 'diffusion_only', 'features': diffusion},
    {'name': 'all_neuroimaging', 'features': structural + diffusion},
    {'name': 'background_structural', 'features': background + structural},
    {'name': 'background_diffusion', 'features': background + diffusion},
    {'name': 'background_structural_diffusion', 'features': background + structural + diffusion},
    {'name': 'structural_ext_only', 'features': structural_ext},
    {'name': 'background_structural_ext', 'features': background + structural_ext},
    {'name': 'background_structural_ext_diffusion', 'features': background + structural_ext + diffusion},
]

print('Combinations:', len(combinations))


Combinations: 10


In [4]:
# Purpose: build long format (visit=1/2) for visit‑classification

def build_long_from_wide(df, subject_col):
    rows = []
    for _, r in df.iterrows():
        sid = r[subject_col]
        # visit 1
        row1 = {subject_col: sid, 'visit': 1}
        row1['age'] = r.get('age1')
        row1['age_at_onset'] = r.get('age_at_onset')
        row1['dur'] = r.get('dur1')
        row1['sex'] = r.get('sex')
        row1['GAA1'] = r.get('GAA1')
        row1['GAA2'] = r.get('GAA2')
        row1['FARS'] = r.get('FARS1')
        row1['SARA'] = r.get('SARA1')

        for f in structural_ext + diffusion:
            col = f + '_v1'
            if col in df.columns:
                row1[f] = r.get(col)

        # visit 2
        row2 = {subject_col: sid, 'visit': 2}
        row2['age'] = r.get('age2')
        row2['age_at_onset'] = r.get('age_at_onset')
        row2['dur'] = r.get('dur2')
        row2['sex'] = r.get('sex')
        row2['GAA1'] = r.get('GAA1')
        row2['GAA2'] = r.get('GAA2')
        row2['FARS'] = r.get('FARS2')
        row2['SARA'] = r.get('SARA2')

        for f in structural_ext + diffusion:
            col = f + '_v2'
            if col in df.columns:
                row2[f] = r.get(col)

        rows.append(row1)
        rows.append(row2)

    return pd.DataFrame(rows)

long_df = build_long_from_wide(melb_wide, subject_col)
print('Long rows:', len(long_df), 'subjects:', long_df[subject_col].nunique())


Long rows: 52 subjects: 26


In [5]:
# Purpose: helpers for CV, scoring, and feature selection

def filter_complete_pairs(df, subject_col, feature_cols):
    cols = [subject_col, 'visit'] + list(feature_cols)
    for c in ['FARS', 'SARA']:
        if c in df.columns:
            cols.append(c)
    sub = df[cols].copy()
    sub = sub.dropna(subset=feature_cols)
    counts = sub.groupby(subject_col)['visit'].nunique()
    valid = counts[counts == 2].index
    return sub[sub[subject_col].isin(valid)].copy()


def compute_paired_d(oof_df, subject_col, visit_col='visit', score_col='score'):
    paired = (oof_df.sort_values([subject_col, visit_col])
                .groupby(subject_col)[score_col]
                .apply(list))
    paired = paired[paired.map(len) == 2]
    if len(paired) < 2:
        return np.nan
    diffs = paired.map(lambda x: x[1] - x[0]).astype(float)
    sd = diffs.std(ddof=1)
    return np.nan if sd == 0 else diffs.mean() / sd


def fit_linear_calibration(x_train, y_train):
    x = np.asarray(x_train).reshape(-1)
    y = np.asarray(y_train).reshape(-1)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return None
    a, b = np.polyfit(x[mask], y[mask], 1)
    return a, b


def apply_linear_calibration(x, calib):
    x = np.asarray(x).reshape(-1)
    if calib is None:
        return np.full(len(x), np.nan)
    a, b = calib
    return a * x + b


def _pearsonr_np(a, b):
    a = np.asarray(a).reshape(-1)
    b = np.asarray(b).reshape(-1)
    if len(a) < 2:
        return np.nan
    if np.std(a) == 0 or np.std(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def compute_regression_metrics(y_true, y_pred, prefix):
    if y_true is None or y_pred is None:
        return {
            f'{prefix}_pearson_r': np.nan,
            f'{prefix}_spearman_rho': np.nan,
            f'{prefix}_rmse': np.nan,
            f'{prefix}_mae': np.nan,
            f'{prefix}_r2': np.nan,
            f'{prefix}_n': 0,
        }
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    n = min(len(yt), len(yp))
    yt = yt[:n]
    yp = yp[:n]
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if len(yt) < 3:
        return {
            f'{prefix}_pearson_r': np.nan,
            f'{prefix}_spearman_rho': np.nan,
            f'{prefix}_rmse': np.nan,
            f'{prefix}_mae': np.nan,
            f'{prefix}_r2': np.nan,
            f'{prefix}_n': int(len(yt)),
        }

    err = yp - yt
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))

    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    r2 = np.nan if ss_tot == 0 else float(1.0 - ss_res / ss_tot)

    pearson_r = _pearsonr_np(yt, yp)
    # scipy-free Spearman via ranks
    yt_rank = pd.Series(yt).rank(method='average').to_numpy()
    yp_rank = pd.Series(yp).rank(method='average').to_numpy()
    spearman_rho = _pearsonr_np(yt_rank, yp_rank)

    return {
        f'{prefix}_pearson_r': pearson_r,
        f'{prefix}_spearman_rho': spearman_rho,
        f'{prefix}_rmse': rmse,
        f'{prefix}_mae': mae,
        f'{prefix}_r2': r2,
        f'{prefix}_n': int(len(yt)),
    }


def get_score(model, X):
    if hasattr(model, 'decision_function'):
        return model.decision_function(X)
    return model.predict_proba(X)[:, 1]


def select_topk_mi(X_train, y_train, feature_names, k):
    k = min(k, len(feature_names))
    mi = mutual_info_classif(X_train, y_train, random_state=RANDOM_SEED)
    idx = np.argsort(mi)[::-1][:k]
    return [feature_names[i] for i in idx]


def inner_cv_select(X, y, groups, feature_names, model_type, use_fs):
    # full grid definitions
    grids = []
    if model_type == 'LOGREG':
        for penalty in ['l1', 'l2']:
            for C in [0.001, 0.01, 0.1, 1, 10, 100]:
                for solver in ['liblinear', 'saga']:
                    grids.append({'penalty': penalty, 'C': C, 'solver': solver})
    elif model_type == 'SVM':
        for kernel in ['linear', 'rbf']:
            for C in [0.001, 0.01, 0.1, 1, 10, 100]:
                if kernel == 'rbf':
                    for gamma in ['scale', 'auto']:
                        grids.append({'kernel': kernel, 'C': C, 'gamma': gamma})
                else:
                    grids.append({'kernel': kernel, 'C': C})
    elif model_type == 'RF':
        for n_estimators in [200, 500, 800]:
            for max_depth in [None, 10, 20, 40]:
                for min_samples_split in [2, 5, 10]:
                    for min_samples_leaf in [1, 2, 4]:
                        grids.append({
                            'n_estimators': n_estimators,
                            'max_depth': max_depth,
                            'min_samples_split': min_samples_split,
                            'min_samples_leaf': min_samples_leaf,
                        })

    best = None
    best_params = None
    best_k = None

    gkf = GroupKFold(n_splits=INNER_FOLDS)
    for params in grids:
        k_list = K_LIST if use_fs else [None]
        for k in k_list:
            scores = []
            for train_idx, val_idx in gkf.split(X, y, groups):
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                feats = feature_names
                if k is not None:
                    feats = select_topk_mi(X_train, y_train, feature_names, k)
                    feat_idx = [feature_names.index(f) for f in feats]
                    X_train = X_train[:, feat_idx]
                    X_val = X_val[:, feat_idx]

                # standardize
                xs = StandardScaler()
                X_train_s = xs.fit_transform(X_train)
                X_val_s = xs.transform(X_val)

                if model_type == 'LOGREG':
                    model = LogisticRegression(max_iter=5000, **params)
                elif model_type == 'SVM':
                    model = SVC(probability=True, **params)
                else:
                    model = RandomForestClassifier(random_state=RANDOM_SEED, **params)

                model.fit(X_train_s, y_train)
                preds = model.predict(X_val_s)
                scores.append(accuracy_score(y_val, preds))

            score = np.mean(scores)
            if best is None or score > best:
                best = score
                best_params = params
                best_k = k

    return best_params, best_k


def permutation_importance(model, X_test, y_test, feature_names):
    base = accuracy_score(y_test, model.predict(X_test))
    imps = []
    rng = np.random.RandomState(RANDOM_SEED)
    for j, f in enumerate(feature_names):
        scores = []
        for _ in range(N_PERM):
            Xp = X_test.copy()
            Xp[:, j] = rng.permutation(Xp[:, j])
            scores.append(accuracy_score(y_test, model.predict(Xp)))
        imps.append((f, base - np.mean(scores)))
    return imps


In [6]:
# Purpose: run full‑feature models (inner 5‑fold tuning, outer LOO)

summary_rows = []
feat_imp_rows = []

model_types = ['LOGREG', 'SVM', 'RF']

for combo in combinations:
    feats = combo['features']
    sub = filter_complete_pairs(long_df, subject_col, feats)
    if sub.empty:
        continue

    X = sub[feats].values
    y = (sub['visit'].values == 2).astype(int)
    groups = sub[subject_col].values

    for mtype in model_types:
        # inner CV tuning on training data only
        best_params, _ = inner_cv_select(X, y, groups, feats, mtype, use_fs=False)

        all_true, all_pred, all_scores = [], [], []
        oof_rows = []

        for sid in np.unique(groups):
            train_mask = groups != sid
            test_mask = groups == sid

            X_train, X_test = X[train_mask], X[test_mask]
            y_train, y_test = y[train_mask], y[test_mask]
            fars_train = sub.loc[train_mask, 'FARS'].values if 'FARS' in sub.columns else np.full(train_mask.sum(), np.nan)
            sara_train = sub.loc[train_mask, 'SARA'].values if 'SARA' in sub.columns else np.full(train_mask.sum(), np.nan)
            fars_true = sub.loc[test_mask, 'FARS'].values if 'FARS' in sub.columns else np.full(test_mask.sum(), np.nan)
            sara_true = sub.loc[test_mask, 'SARA'].values if 'SARA' in sub.columns else np.full(test_mask.sum(), np.nan)

            xs = StandardScaler()
            X_train_s = xs.fit_transform(X_train)
            X_test_s = xs.transform(X_test)

            if mtype == 'LOGREG':
                model = LogisticRegression(max_iter=5000, **best_params)
            elif mtype == 'SVM':
                model = SVC(probability=True, **best_params)
            else:
                model = RandomForestClassifier(random_state=RANDOM_SEED, **best_params)

            model.fit(X_train_s, y_train)
            preds = model.predict(X_test_s)
            scores_train = get_score(model, X_train_s)
            scores = get_score(model, X_test_s)

            calib_fars = fit_linear_calibration(scores_train, fars_train)
            calib_sara = fit_linear_calibration(scores_train, sara_train)
            fars_pred = apply_linear_calibration(scores, calib_fars)
            sara_pred = apply_linear_calibration(scores, calib_sara)

            all_true.extend(y_test)
            all_pred.extend(preds)
            all_scores.extend(scores)

            for v, sc, f_t, s_t, f_p, s_p in zip(sub.loc[test_mask, 'visit'].values, scores, fars_true, sara_true, fars_pred, sara_pred):
                oof_rows.append({'subject': sid, 'visit': v, 'score': sc, 'FARS': f_t, 'SARA': s_t, 'FARS_pred': f_p, 'SARA_pred': s_p})

            # permutation importance per fold
            imps = permutation_importance(model, X_test_s, y_test, feats)
            for f, imp in imps:
                feat_imp_rows.append({
                    'combination': combo['name'],
                    'model': mtype,
                    'task': 'classification',
                    'feature_set': 'full',
                    'feature': f,
                    'importance_type': 'permutation',
                    'importance_value': imp,
                    'n_subjects': sub[subject_col].nunique(),
                })

            # model-based importance
            if hasattr(model, 'coef_'):
                for f, imp in zip(feats, model.coef_.ravel()):
                    feat_imp_rows.append({
                        'combination': combo['name'],
                        'model': mtype,
                        'task': 'classification',
                        'feature_set': 'full',
                        'feature': f,
                        'importance_type': 'coef',
                        'importance_value': imp,
                        'n_subjects': sub[subject_col].nunique(),
                    })
            if hasattr(model, 'feature_importances_'):
                for f, imp in zip(feats, model.feature_importances_):
                    feat_imp_rows.append({
                        'combination': combo['name'],
                        'model': mtype,
                        'task': 'classification',
                        'feature_set': 'full',
                        'feature': f,
                        'importance_type': 'impurity',
                        'importance_value': imp,
                        'n_subjects': sub[subject_col].nunique(),
                    })

        acc = accuracy_score(all_true, all_pred)
        oof_df = pd.DataFrame(oof_rows)
        d_score = compute_paired_d(oof_df, 'subject')

        fars_metrics = compute_regression_metrics(oof_df.get('FARS'), oof_df.get('FARS_pred'), 'fars')
        sara_metrics = compute_regression_metrics(oof_df.get('SARA'), oof_df.get('SARA_pred'), 'sara')

        # delta metrics (visit2 - visit1), one row per subject
        df_sorted = oof_df.sort_values(['subject', 'visit']).copy()
        delta_rows = []
        for sid2, g in df_sorted.groupby('subject'):
            if len(g) != 2:
                continue
            v = g['visit'].values
            if set(v) != {1, 2}:
                continue
            g = g.sort_values('visit')
            delta_rows.append({
                'subject': sid2,
                'FARS_delta_true': float(g['FARS'].values[1] - g['FARS'].values[0]) if 'FARS' in g else np.nan,
                'FARS_delta_pred': float(g['FARS_pred'].values[1] - g['FARS_pred'].values[0]) if 'FARS_pred' in g else np.nan,
                'SARA_delta_true': float(g['SARA'].values[1] - g['SARA'].values[0]) if 'SARA' in g else np.nan,
                'SARA_delta_pred': float(g['SARA_pred'].values[1] - g['SARA_pred'].values[0]) if 'SARA_pred' in g else np.nan,
            })
        delta_df = pd.DataFrame(delta_rows)
        fars_delta_metrics = compute_regression_metrics(delta_df.get('FARS_delta_true'), delta_df.get('FARS_delta_pred'), 'fars_delta')
        sara_delta_metrics = compute_regression_metrics(delta_df.get('SARA_delta_true'), delta_df.get('SARA_delta_pred'), 'sara_delta')

        summary_rows.append({
            'combination': combo['name'],
            'model': mtype,
            'feature_set': 'full',
            'accuracy': acc,
            'cohen_d': d_score,
            'best_params': str(best_params),
            **fars_metrics,
            **sara_metrics,
            **fars_delta_metrics,
            **sara_delta_metrics,
        })

full_results_df = pd.DataFrame(summary_rows)
full_results_df.sort_values(['accuracy'], ascending=False).head(10)


,combination,model,feature_set,accuracy,cohen_d,best_params,fars_pearson_r,fars_spearman_rho,fars_rmse,fars_mae,...,fars_delta_rmse,fars_delta_mae,fars_delta_r2,fars_delta_n,sara_delta_pearson_r,sara_delta_spearman_rho,sara_delta_rmse,sara_delta_mae,sara_delta_r2,sara_delta_n
3,structural_only,LOGREG,full,0.692308,0.551595,"{'penalty': 'l1', 'C': 10, 'solver': 'liblinear'}",-0.001055,0.019939,27.113895,22.473489,...,10.833157,7.841166,-0.936662,26,0.247645,0.319466,3.286344,2.531079,-0.433324,23
18,background_structural_diffusion,LOGREG,full,0.673077,0.789231,"{'penalty': 'l1', 'C': 1, 'solver': 'liblinear'}",0.059996,0.078434,26.765596,21.586844,...,10.836433,8.362457,-0.937834,26,0.242048,0.277861,3.154017,2.417001,-0.320221,23
4,structural_only,SVM,full,0.634615,0.531891,"{'kernel': 'linear', 'C': 10}",-0.184847,-0.164468,27.519090,22.829950,...,10.500298,7.842051,-0.819479,26,0.251762,0.314018,3.128782,2.485069,-0.299179,23
28,background_structural_ext_diffusion,SVM,full,0.634615,0.430060,"{'kernel': 'rbf', 'C': 100, 'gamma': 'scale'}",-0.129345,-0.105888,26.745161,22.137801,...,10.306866,7.715290,-0.753061,26,0.221865,0.271918,3.222725,2.701392,-0.378367,23
17,background_diffusion,RF,full,0.634615,0.231023,"{'n_estimators': 500, 'max_depth': None, 'min_...",-0.483298,-0.493660,27.259994,22.894130,...,10.368643,7.792043,-0.774139,26,0.290145,0.308074,3.370677,2.923401,-0.507831,23
2,background_only,RF,full,0.615385,0.522941,"{'n_estimators': 200, 'max_depth': None, 'min_...",0.167296,0.303245,26.255398,21.655873,...,12.001462,8.746234,-1.376907,26,0.190516,0.209562,3.486888,2.738559,-0.613595,23
9,all_neuroimaging,LOGREG,full,0.615385,0.403223,"{'penalty': 'l1', 'C': 10, 'solver': 'liblinear'}",0.131974,0.169719,27.263267,22.101062,...,13.041598,10.200728,-1.806761,26,0.283093,0.332344,3.657246,2.856687,-0.775115,23
22,structural_ext_only,SVM,full,0.615385,0.487050,"{'kernel': 'rbf', 'C': 100, 'gamma': 'scale'}",-0.248313,-0.155075,27.404156,22.525672,...,10.293307,7.196141,-0.748452,26,-0.097294,-0.046063,3.437391,2.845450,-0.568109,23
11,all_neuroimaging,RF,full,0.596154,0.319885,"{'n_estimators': 200, 'max_depth': None, 'min_...",-0.462999,-0.242902,27.130957,22.492170,...,10.212295,7.600760,-0.721038,26,0.313358,0.209015,3.316134,2.862691,-0.459428,23
19,background_structural_diffusion,SVM,full,0.596154,0.439287,"{'kernel': 'rbf', 'C': 10, 'gamma': 'scale'}",-0.091525,-0.114427,26.660493,22.266178,...,10.347133,7.781766,-0.766786,26,0.119789,0.148094,3.284216,2.765791,-0.431469,23


In [7]:
# run entropy‑selected models (inner 5‑fold tuning, outer LOO)

summary_rows = []

for combo in combinations:
    feats = combo['features']
    sub = filter_complete_pairs(long_df, subject_col, feats)
    if sub.empty:
        continue

    X = sub[feats].values
    y = (sub['visit'].values == 2).astype(int)
    groups = sub[subject_col].values

    for mtype in model_types:
        best_params, best_k = inner_cv_select(X, y, groups, feats, mtype, use_fs=True)

        all_true, all_pred, all_scores = [], [], []
        oof_rows = []

        for sid in np.unique(groups):
            train_mask = groups != sid
            test_mask = groups == sid

            X_train, X_test = X[train_mask], X[test_mask]
            y_train, y_test = y[train_mask], y[test_mask]
            fars_train = sub.loc[train_mask, 'FARS'].values if 'FARS' in sub.columns else np.full(train_mask.sum(), np.nan)
            sara_train = sub.loc[train_mask, 'SARA'].values if 'SARA' in sub.columns else np.full(train_mask.sum(), np.nan)
            fars_true = sub.loc[test_mask, 'FARS'].values if 'FARS' in sub.columns else np.full(test_mask.sum(), np.nan)
            sara_true = sub.loc[test_mask, 'SARA'].values if 'SARA' in sub.columns else np.full(test_mask.sum(), np.nan)

            # select top‑K by MI on training only
            feats_k = select_topk_mi(X_train, y_train, feats, best_k)
            feat_idx = [feats.index(f) for f in feats_k]
            X_train = X_train[:, feat_idx]
            X_test = X_test[:, feat_idx]

            xs = StandardScaler()
            X_train_s = xs.fit_transform(X_train)
            X_test_s = xs.transform(X_test)

            if mtype == 'LOGREG':
                model = LogisticRegression(max_iter=5000, **best_params)
            elif mtype == 'SVM':
                model = SVC(probability=True, **best_params)
            else:
                model = RandomForestClassifier(random_state=RANDOM_SEED, **best_params)

            model.fit(X_train_s, y_train)
            preds = model.predict(X_test_s)
            scores_train = get_score(model, X_train_s)
            scores = get_score(model, X_test_s)

            calib_fars = fit_linear_calibration(scores_train, fars_train)
            calib_sara = fit_linear_calibration(scores_train, sara_train)
            fars_pred = apply_linear_calibration(scores, calib_fars)
            sara_pred = apply_linear_calibration(scores, calib_sara)

            all_true.extend(y_test)
            all_pred.extend(preds)
            all_scores.extend(scores)

            for v, sc, f_t, s_t, f_p, s_p in zip(sub.loc[test_mask, 'visit'].values, scores, fars_true, sara_true, fars_pred, sara_pred):
                oof_rows.append({'subject': sid, 'visit': v, 'score': sc, 'FARS': f_t, 'SARA': s_t, 'FARS_pred': f_p, 'SARA_pred': s_p})

            # permutation importance per fold
            imps = permutation_importance(model, X_test_s, y_test, feats_k)
            for f, imp in imps:
                feat_imp_rows.append({
                    'combination': combo['name'],
                    'model': mtype,
                    'task': 'classification',
                    'feature_set': 'entropy',
                    'feature': f,
                    'importance_type': 'permutation',
                    'importance_value': imp,
                    'n_subjects': sub[subject_col].nunique(),
                })

            # model-based importance
            if hasattr(model, 'coef_'):
                for f, imp in zip(feats_k, model.coef_.ravel()):
                    feat_imp_rows.append({
                        'combination': combo['name'],
                        'model': mtype,
                        'task': 'classification',
                        'feature_set': 'entropy',
                        'feature': f,
                        'importance_type': 'coef',
                        'importance_value': imp,
                        'n_subjects': sub[subject_col].nunique(),
                    })
            if hasattr(model, 'feature_importances_'):
                for f, imp in zip(feats_k, model.feature_importances_):
                    feat_imp_rows.append({
                        'combination': combo['name'],
                        'model': mtype,
                        'task': 'classification',
                        'feature_set': 'entropy',
                        'feature': f,
                        'importance_type': 'impurity',
                        'importance_value': imp,
                        'n_subjects': sub[subject_col].nunique(),
                    })

        acc = accuracy_score(all_true, all_pred)
        oof_df = pd.DataFrame(oof_rows)
        d_score = compute_paired_d(oof_df, 'subject')

        fars_metrics = compute_regression_metrics(oof_df.get('FARS'), oof_df.get('FARS_pred'), 'fars')
        sara_metrics = compute_regression_metrics(oof_df.get('SARA'), oof_df.get('SARA_pred'), 'sara')

        df_sorted = oof_df.sort_values(['subject', 'visit']).copy()
        delta_rows = []
        for sid2, g in df_sorted.groupby('subject'):
            if len(g) != 2:
                continue
            v = g['visit'].values
            if set(v) != {1, 2}:
                continue
            g = g.sort_values('visit')
            delta_rows.append({
                'subject': sid2,
                'FARS_delta_true': float(g['FARS'].values[1] - g['FARS'].values[0]) if 'FARS' in g else np.nan,
                'FARS_delta_pred': float(g['FARS_pred'].values[1] - g['FARS_pred'].values[0]) if 'FARS_pred' in g else np.nan,
                'SARA_delta_true': float(g['SARA'].values[1] - g['SARA'].values[0]) if 'SARA' in g else np.nan,
                'SARA_delta_pred': float(g['SARA_pred'].values[1] - g['SARA_pred'].values[0]) if 'SARA_pred' in g else np.nan,
            })
        delta_df = pd.DataFrame(delta_rows)
        fars_delta_metrics = compute_regression_metrics(delta_df.get('FARS_delta_true'), delta_df.get('FARS_delta_pred'), 'fars_delta')
        sara_delta_metrics = compute_regression_metrics(delta_df.get('SARA_delta_true'), delta_df.get('SARA_delta_pred'), 'sara_delta')

        summary_rows.append({
            'combination': combo['name'],
            'model': mtype,
            'feature_set': 'entropy',
            'accuracy': acc,
            'cohen_d': d_score,
            'best_params': str(best_params),
            'best_k': best_k,
            **fars_metrics,
            **sara_metrics,
            **fars_delta_metrics,
            **sara_delta_metrics,
        })

entropy_results_df = pd.DataFrame(summary_rows)
entropy_results_df.sort_values(['accuracy'], ascending=False).head(10)


,combination,model,feature_set,accuracy,cohen_d,best_params,best_k,fars_pearson_r,fars_spearman_rho,fars_rmse,...,fars_delta_rmse,fars_delta_mae,fars_delta_r2,fars_delta_n,sara_delta_pearson_r,sara_delta_spearman_rho,sara_delta_rmse,sara_delta_mae,sara_delta_r2,sara_delta_n
3,structural_only,LOGREG,entropy,0.692308,0.551577,"{'penalty': 'l1', 'C': 10, 'solver': 'liblinear'}",500,-0.001028,0.019939,27.113740,...,10.833478,7.841508,-0.936777,26,0.247618,0.319466,3.286473,2.531251,-0.433436,23
18,background_structural_diffusion,LOGREG,entropy,0.673077,0.789282,"{'penalty': 'l1', 'C': 1, 'solver': 'liblinear'}",500,0.059964,0.078434,26.765782,...,10.836326,8.362383,-0.937796,26,0.242003,0.277861,3.154068,2.416999,-0.320263,23
4,structural_only,SVM,entropy,0.634615,0.531891,"{'kernel': 'linear', 'C': 10}",500,-0.184847,-0.164468,27.519090,...,10.500298,7.842051,-0.819479,26,0.251762,0.314018,3.128782,2.485069,-0.299179,23
28,background_structural_ext_diffusion,SVM,entropy,0.634615,0.430060,"{'kernel': 'rbf', 'C': 100, 'gamma': 'scale'}",500,-0.129345,-0.105888,26.745161,...,10.306866,7.715290,-0.753061,26,0.221865,0.271918,3.222725,2.701392,-0.378367,23
17,background_diffusion,RF,entropy,0.615385,0.322664,"{'n_estimators': 200, 'max_depth': None, 'min_...",500,-0.469923,-0.464711,27.248865,...,10.413363,7.916365,-0.789476,26,0.179337,0.177316,3.364501,2.931523,-0.502310,23
22,structural_ext_only,SVM,entropy,0.615385,0.487050,"{'kernel': 'rbf', 'C': 100, 'gamma': 'scale'}",500,-0.248313,-0.155075,27.404156,...,10.293307,7.196141,-0.748452,26,-0.097294,-0.046063,3.437391,2.845450,-0.568109,23
9,all_neuroimaging,LOGREG,entropy,0.615385,0.403323,"{'penalty': 'l1', 'C': 10, 'solver': 'liblinear'}",500,0.132011,0.169719,27.262655,...,13.041032,10.200521,-1.806517,26,0.283190,0.332344,3.656933,2.856579,-0.774812,23
29,background_structural_ext_diffusion,RF,entropy,0.615385,0.476381,"{'n_estimators': 800, 'max_depth': None, 'min_...",500,-0.084640,-0.007856,26.576826,...,10.284524,7.644312,-0.745469,26,0.172044,0.145617,3.241343,2.851158,-0.394339,23
27,background_structural_ext_diffusion,LOGREG,entropy,0.615385,0.690721,"{'penalty': 'l2', 'C': 0.001, 'solver': 'saga'}",5000,0.257362,0.286879,25.575018,...,11.789701,8.932840,-1.293768,26,0.332142,0.230808,3.434468,2.935815,-0.565444,23
7,diffusion_only,SVM,entropy,0.596154,0.142110,"{'kernel': 'rbf', 'C': 0.001, 'gamma': 'scale'}",500,-0.352718,-0.358140,27.351797,...,10.716245,8.283412,-0.895087,26,-0.190665,-0.044577,3.850138,3.097188,-0.967301,23


In [8]:
# final summary display (accuracy + Cohen's d)

summary_df = pd.concat([full_results_df, entropy_results_df], ignore_index=True)
summary_df = summary_df.sort_values(['accuracy'], ascending=False)
cols = ['model','feature_set','combination','accuracy','cohen_d','fars_r2','fars_rmse','fars_n','sara_r2','sara_rmse','sara_n','fars_delta_r2','fars_delta_rmse','fars_delta_n','sara_delta_r2','sara_delta_rmse','sara_delta_n','best_params','best_k']
summary_df.reindex(columns=cols).head(20)


,model,feature_set,combination,accuracy,cohen_d,fars_r2,fars_rmse,fars_n,sara_r2,sara_rmse,sara_n,fars_delta_r2,fars_delta_rmse,fars_delta_n,sara_delta_r2,sara_delta_rmse,sara_delta_n,best_params,best_k
3,LOGREG,full,structural_only,0.692308,0.551595,-0.067294,27.113895,52,-0.039078,8.395062,49,-0.936662,10.833157,26,-0.433324,3.286344,23,"{'penalty': 'l1', 'C': 10, 'solver': 'liblinear'}",NaN
33,LOGREG,entropy,structural_only,0.692308,0.551577,-0.067282,27.113740,52,-0.039068,8.395021,49,-0.936777,10.833478,26,-0.433436,3.286473,23,"{'penalty': 'l1', 'C': 10, 'solver': 'liblinear'}",500.0
48,LOGREG,entropy,background_structural_diffusion,0.673077,0.789282,-0.040064,26.765782,52,-0.035232,8.379512,49,-0.937796,10.836326,26,-0.320263,3.154068,23,"{'penalty': 'l1', 'C': 1, 'solver': 'liblinear'}",500.0
18,LOGREG,full,background_structural_diffusion,0.673077,0.789231,-0.040050,26.765596,52,-0.035218,8.379456,49,-0.937834,10.836433,26,-0.320221,3.154017,23,"{'penalty': 'l1', 'C': 1, 'solver': 'liblinear'}",NaN
17,RF,full,background_diffusion,0.634615,0.231023,-0.078827,27.259994,52,-0.076503,8.544910,49,-0.774139,10.368643,26,-0.507831,3.370677,23,"{'n_estimators': 500, 'max_depth': None, 'min_...",NaN
4,SVM,full,structural_only,0.634615,0.531891,-0.099432,27.519090,52,-0.087760,8.589470,49,-0.819479,10.500298,26,-0.299179,3.128782,23,"{'kernel': 'linear', 'C': 10}",NaN
58,SVM,entropy,background_structural_ext_diffusion,0.634615,0.430060,-0.038462,26.745161,52,-0.032045,8.366604,49,-0.753061,10.306866,26,-0.378367,3.222725,23,"{'kernel': 'rbf', 'C': 100, 'gamma': 'scale'}",500.0
34,SVM,entropy,structural_only,0.634615,0.531891,-0.099432,27.519090,52,-0.087760,8.589470,49,-0.819479,10.500298,26,-0.299179,3.128782,23,"{'kernel': 'linear', 'C': 10}",500.0
28,SVM,full,background_structural_ext_diffusion,0.634615,0.430060,-0.038462,26.745161,52,-0.032045,8.366604,49,-0.753061,10.306866,26,-0.378367,3.222725,23,"{'kernel': 'rbf', 'C': 100, 'gamma': 'scale'}",NaN
47,RF,entropy,background_diffusion,0.615385,0.322664,-0.077946,27.248865,52,-0.073029,8.531109,49,-0.789476,10.413363,26,-0.502310,3.364501,23,"{'n_estimators': 200, 'max_depth': None, 'min_...",500.0


In [9]:
# save single feature-importance CSV

feature_importance_df = pd.DataFrame(feat_imp_rows)
feature_importance_df.to_csv(RESULTS_DIR / 'feature_importance_ml_extensions.csv', index=False)
print('Saved:', RESULTS_DIR / 'feature_importance_ml_extensions.csv')

feature_importance_df.head()


Saved: results/feature_importance_ml_extensions.csv


,combination,model,task,feature_set,feature,importance_type,importance_value,n_subjects
0,background_only,LOGREG,classification,full,age,permutation,-0.35,26
1,background_only,LOGREG,classification,full,age_at_onset,permutation,0.00,26
2,background_only,LOGREG,classification,full,dur,permutation,0.05,26
3,background_only,LOGREG,classification,full,sex,permutation,0.00,26
4,background_only,LOGREG,classification,full,GAA1,permutation,0.00,26
